# 02 - Feature Selection RFE and Impact
Seleccion de variables con RFECV y evaluacion de impacto por grupos.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.feature_selection import RFECV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings("ignore")

dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "data" / "processed" / "dataset_entrenamiento_final.csv"
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError("dataset_entrenamiento_final.csv not found under data/processed")

df = pd.read_csv(dataset_path, parse_dates=["date"])
print("dataset:", dataset_path.resolve())
print("shape:", df.shape)

blacklist = [
    "precio_provincial_lag_1"
    "precio_provincial_lag_2"
    "precio_provincial_lag_3"
    "precio_vecinos_media_lag1"
    "precio_nacional_base_ma3"
    "precio_nacional_base_ma6"
    "precio_nacional_base_vol3"
    "precio_nacional_base_vol6"
]
target_candidates = ["precio_provincial_TARGET_H1", "precio_provincial_TARGET_H2", "precio_provincial_TARGET_H3"]
missing_targets = [t for t in target_candidates if t not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")
available_targets = target_candidates
horizons = [1, 2, 3]

split_date = pd.Timestamp("2021-01-01")
train_mask = df["date"] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()
print("train rows:", train_df.shape[0], "test rows:", test_df.shape[0])

identifiers = ["date", "provincia", "cereal_predominante"]
training_cols = get_training_features(df)
feature_cols = [
    c for c in training_cols
    if c in df.columns and c not in identifiers + available_targets
]
feature_cols = [c for c in feature_cols if c not in blacklist]

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

base_price_col = "precio_provincial_lag_1"
if base_price_col not in df.columns:
    raise ValueError("precio_provincial_lag_1 missing for targets")

def build_targets(horizon: int):
    target_reg = f"precio_provincial_TARGET_H{horizon}"
    y_train_reg = train_df[target_reg]
    y_test_reg = test_df[target_reg]
    base_train = train_df[base_price_col]
    base_test = test_df[base_price_col]
    return y_train_reg, y_test_reg, base_train, base_test

def regression_metrics(y_true, y_pred):
    aligned = pd.concat([y_true, y_pred], axis=1).dropna()
    if aligned.empty:
        return {"MAE": np.nan, "RMSE": np.nan, "Pearson": np.nan}
    y_true_clean = aligned.iloc[:, 0]
    y_pred_clean = aligned.iloc[:, 1]
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    pearson = pearsonr(y_true_clean, y_pred_clean)[0] if y_true_clean.nunique() > 1 else np.nan
    return {"MAE": float(mae), "RMSE": float(rmse), "Pearson": float(pearson) if pearson == pearson else np.nan}

def eval_model(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = pd.Series(model.predict(X_te), index=y_te.index)
    return regression_metrics(y_te, preds)

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 78)
train rows: 5394 test rows: 1653


## 1. RFECV con XGBoost regularizado
Seleccion de features entre 10 y 76 usando parametros de regularizacion del notebook anterior.

In [2]:
tscv = TimeSeriesSplit(n_splits=5)

xgb_base = XGBRegressor(
    random_state=42,
    objective="reg:squarederror",
    learning_rate=0.05,
    subsample=0.7,
    reg_lambda=3,
)

selector = RFECV(
    estimator=xgb_base,
    min_features_to_select=10,
    scoring="neg_mean_absolute_error",
)

h = 1
y_train_reg, y_test_reg, base_train, base_test = build_targets(h)
train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
X_train_h = X_train.loc[train_mask_h]
X_test_h = X_test.loc[test_mask_h]
base_train_h = base_train.loc[train_mask_h]
base_test_h = base_test.loc[test_mask_h]
y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

selector.fit(X_train_h, y_train_ret)
selected_mask = selector.support_
selected_features = X_train_h.columns[selected_mask].tolist()
print("Selected features:", len(selected_features))

rfe_curve = pd.DataFrame({
    "n_features": selector.cv_results_["n_features"],
    "score": selector.cv_results_["mean_test_score"],
})
rfe_curve.head()

Selected features: 76


,n_features,score
0,10,-0.056484
1,11,-0.050335
2,12,-0.049118
3,13,-0.047533
4,14,-0.048483


## 2. Impacto de grupos de variables
Quita grupos (calendario, superficie) manteniendo Urea/Trigo/DAP protegidas.

In [4]:
protected_keywords = ["urea", "wheat", "dap"]

def protect_features(cols):
    keep = []
    for c in cols:
        if any(k in c for k in protected_keywords):
            keep.append(c)
    return keep

groups = {
    "calendario": [c for c in X_train.columns if c.startswith("month_") or c.endswith("_cos") or c.endswith("_sin")],
    "superficie": [c for c in X_train.columns if "sup" in c or "superficie" in c],
}

group_results = []

for group_name, group_cols in groups.items():
    for h in horizons:
        y_train_reg, y_test_reg, base_train, base_test = build_targets(h)
        train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
        test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
        X_train_h = X_train.loc[train_mask_h]
        X_test_h = X_test.loc[test_mask_h]
        base_train_h = base_train.loc[train_mask_h]
        base_test_h = base_test.loc[test_mask_h]
        y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
        y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

        protected = protect_features(X_train_h.columns)
        reduced_cols = [c for c in X_train_h.columns if c not in group_cols or c in protected]

        model = XGBRegressor(
            random_state=42,
            n_jobs=-1,
            objective="reg:squarederror",
            n_estimators=400,
            learning_rate=0.05,
            max_depth=2,
            subsample=0.7,
            colsample_bytree=0.9,
            reg_lambda=3,
            reg_alpha=0.5,
        )

        metrics = eval_model(model, X_train_h[reduced_cols], y_train_ret, X_test_h[reduced_cols], y_test_ret)
        group_results.append({
            "horizon": h,
            "group_removed": group_name,
            "n_features": len(reduced_cols),
            **metrics,
        })

group_results_df = pd.DataFrame(group_results)
group_results_df

,horizon,group_removed,n_features,MAE,RMSE,Pearson
0,1,calendario,75,0.049819,0.069663,0.511245
1,2,calendario,75,0.069245,0.099663,0.610208
2,3,calendario,75,0.093661,0.133271,0.255697
3,1,superficie,71,0.048826,0.068475,0.517855
4,2,superficie,71,0.071165,0.103614,0.387540
5,3,superficie,71,0.094817,0.134619,0.263139


## 3. Comparativa 76 vs Top K
Compara Pearson y MAE anti-crisis con el subconjunto RFE.

In [6]:
compare_rows = []

crisis_mask = (test_df["date"] >= pd.Timestamp("2022-01-01")) & (test_df["date"] <= pd.Timestamp("2022-06-30"))

for h in horizons:
    y_train_reg, y_test_reg, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]
    y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h
    crisis_idx = y_test_ret.index.intersection(test_df.loc[crisis_mask].index)

    model_full = XGBRegressor(
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=2,
        subsample=0.7,
        colsample_bytree=0.9,
        reg_lambda=3,
        reg_alpha=0.5,
    )
    model_full.fit(X_train_h, y_train_ret)
    preds_full = pd.Series(model_full.predict(X_test_h), index=y_test_ret.index)
    metrics_full = regression_metrics(y_test_ret, preds_full)
    mae_crisis_full = mean_absolute_error(y_test_ret.loc[crisis_idx], preds_full.loc[crisis_idx])

    X_train_k = X_train_h[selected_features]
    X_test_k = X_test_h[selected_features]
    model_k = XGBRegressor(
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=2,
        subsample=0.7,
        colsample_bytree=0.9,
        reg_lambda=3,
        reg_alpha=0.5,
    )
    model_k.fit(X_train_k, y_train_ret)
    preds_k = pd.Series(model_k.predict(X_test_k), index=y_test_ret.index)
    metrics_k = regression_metrics(y_test_ret, preds_k)
    mae_crisis_k = mean_absolute_error(y_test_ret.loc[crisis_idx], preds_k.loc[crisis_idx])

    compare_rows.append({
        "horizon": h,
        "features_full": X_train_h.shape[1],
        "features_k": len(selected_features),
        "Pearson_full": metrics_full["Pearson"],
        "Pearson_k": metrics_k["Pearson"],
        "MAE_crisis_full": mae_crisis_full,
        "MAE_crisis_k": mae_crisis_k,
    })

compare_df = pd.DataFrame(compare_rows)
compare_df

,horizon,features_full,features_k,Pearson_full,Pearson_k,MAE_crisis_full,MAE_crisis_k
0,1,77,76,0.508908,0.574424,0.105081,0.097562
1,2,77,76,0.535772,0.524708,0.158262,0.161925
2,3,77,76,0.286620,0.320615,0.216131,0.213647


## 4. Reporte
Guardar REDUCCION_DIMENSIONALIDAD_V1.md con listas de variables.

In [7]:
report_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "reports"
    if candidate.exists():
        report_root = candidate
        break
if report_root is None:
    report_root = base_dir / "reports"
    report_root.mkdir(parents=True, exist_ok=True)

report_path = report_root / "REDUCCION_DIMENSIONALIDAD_V1.md"

lines = [
    "# REDUCCION_DIMENSIONALIDAD_V1",
    "",
    "## Resumen",
    "RFECV con XGBoost regularizado y analisis por grupos.",
    "",
    "## Curva RFECV (features vs score)",
    rfe_curve.to_markdown(index=False),
    "",
    "## Variables imprescindibles (Top K)",
    pd.DataFrame({"feature": selected_features}).to_markdown(index=False),
    "",
    "## Impacto por grupos",
    group_results_df.to_markdown(index=False),
    "",
    "## Comparativa full vs Top K",
    compare_df.to_markdown(index=False),
    "",
]

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Reporte guardado en:", report_path.resolve())

Reporte guardado en: C:\Users\marco\Desktop\Repos\DATAGIA-21\reports\REDUCCION_DIMENSIONALIDAD_V1.md
